# 🫀 CV Intel RAG — Notebook 2: Demo & Visualization

**For sales demos.** Loads the pre-built index from Google Drive and walks
through the RAG pipeline step-by-step with charts and timing breakdowns.

**Time to run all:** ~60 seconds (depends on Typhoon API latency)

> Prerequisite: you must have run `01_ingest_and_index.ipynb` first.

### What this notebook shows

1. 📈 Corpus overview — sources, categories, dates
2. 🔎 Live retrieval — query → embedding → top-k chunks with scores
3. 💬 5 demo questions — full RAG answers with [S#] citations
4. ⏱ Latency breakdown — embed / retrieve / LLM


In [ ]:
!pip install -q \
  pydantic pydantic-settings sqlalchemy httpx openai \
  chromadb sentence-transformers rank-bm25 tiktoken \


In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['TYPHOON_API_KEY'] = userdata.get('TYPHOON_API_KEY')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/cv-intel-rag')
DB_PATH     = PROJECT_DIR / 'data' / 'cv_intel.db'
CHROMA_PATH = PROJECT_DIR / 'data' / 'chroma'
assert DB_PATH.exists(),    f'❌ DB not found at {DB_PATH}  — run notebook 1 first'
assert CHROMA_PATH.exists(), f'❌ Chroma not found at {CHROMA_PATH}'


In [ ]:
# ⚠️  Set your GitHub username here before running this cell
GITHUB_USERNAME = 'siriponsri'

import os
%cd /content
import subprocess
subprocess.run(['rm', '-rf', 'cv-intel-rag'], check=False)
!git clone -q https://github.com/{GITHUB_USERNAME}/cv-intel-rag.git
%cd /content/cv-intel-rag

# Make `import src` work in this notebook and in any subprocess
os.environ['PYTHONPATH'] = '/content/cv-intel-rag'
import sys
if '/content/cv-intel-rag' not in sys.path:
    sys.path.insert(0, '/content/cv-intel-rag')

env = f'''APP_ENV=development
DATABASE_URL=sqlite:///{DB_PATH}
CHROMA_PATH={CHROMA_PATH}
DEFAULT_LLM_PROVIDER=typhoon_api
TYPHOON_BASE_URL=https://api.opentyphoon.ai/v1
TYPHOON_MODEL=typhoon-v2.5-30b-a3b-instruct
EMBED_MODEL_NAME=BAAI/bge-m3
EMBED_DEVICE=cuda
'''
open('.env', 'w').write(env)
print('✓ .env written, PYTHONPATH set')


## Step 2 — 📈 Corpus overview



In [ ]:
import sqlite3, matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid')

conn = sqlite3.connect(DB_PATH)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# By source
rows = conn.execute('SELECT source_name, COUNT(*) c FROM records GROUP BY 1 ORDER BY c DESC').fetchall()
axes[0].barh([r[0] for r in rows], [r[1] for r in rows], color='#c0392b')
axes[0].set_title(f'Records by source  (Σ={sum(r[1] for r in rows)})')
axes[0].invert_yaxis()

# By category
rows = conn.execute('SELECT category, COUNT(*) c FROM records GROUP BY 1').fetchall()
axes[1].pie([r[1] for r in rows], labels=[r[0] for r in rows], autopct='%1.0f%%',
            colors=plt.cm.Reds_r([0.2, 0.4, 0.55, 0.7, 0.85][:len(rows)]))
axes[1].set_title('Records by category')

# Over time
rows = conn.execute("""SELECT strftime('%Y-%m', published_date) m, COUNT(*) c
                        FROM records WHERE published_date IS NOT NULL
                        GROUP BY 1 ORDER BY 1""").fetchall()
axes[2].plot([r[0] for r in rows], [r[1] for r in rows], marker='o', color='#c0392b')
axes[2].set_title('Records by month'); axes[2].tick_params(axis='x', rotation=45)



## Step 3 — 🔎 Retrieval visualization

We'll follow ONE query through the pipeline and show exactly what happens


In [ ]:
QUERY = 'ยา SGLT2 ลดความเสี่ยงหัวใจล้มเหลวได้อย่างไร'   # Thai query
# QUERY = 'What are recent FDA safety alerts for statins?'

from src.rag.embedder import get_embedder
from src.rag.retriever import HybridRetriever
import time

embedder = get_embedder()

# ─ Stage 1: embed the query ────────────────────────
t0 = time.time()
vec = embedder.encode_single(QUERY)
t_embed = time.time() - t0

print(f'Query:     {QUERY}')
print(f'Embedding: dim={len(vec)}  time={t_embed*1000:.0f} ms')


In [ ]:
# ─ Stage 2: hybrid retrieve ──────────────────────────
retriever = HybridRetriever(top_k=5)
t0 = time.time()
hits = retriever.retrieve(QUERY)
t_retrieve = time.time() - t0

print(f'Retrieved {len(hits)} chunks in {t_retrieve*1000:.0f} ms\n')

# score breakdown chart
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3))
titles = [(h.metadata.get('title','')[:55]+'…') if len(h.metadata.get('title',''))>55 else h.metadata.get('title','(no title)') for h in hits]
scores = [h.score for h in hits]
ax.barh(titles, scores, color='#c0392b')
ax.invert_yaxis(); ax.set_xlabel('hybrid score (dense 0.6 + BM25 0.4)')
ax.set_title(f'Top-{len(hits)} retrieved chunks for this query')
plt.tight_layout(); plt.show()

for i, h in enumerate(hits, 1):
    print(f'[S{i}] {h.metadata.get("source_name","?")}  score={h.score:.3f}')


## Step 4 — 💬 Full RAG answers (5 demo questions)



In [ ]:
from src.agent.rag_agent import RAGAgent
import time

agent = RAGAgent()

DEMO_QUERIES = [
    'ยา SGLT2 ลดความเสี่ยงหัวใจล้มเหลวในผู้ป่วยเบาหวานได้อย่างไร',
    'What are the latest FDA safety alerts for cardiovascular drugs?',
    'เปรียบเทียบ empagliflozin กับ dapagliflozin ในผู้ป่วย CKD',
    'What phase 3 trials are running for GLP-1 agonists in heart failure?',
    'ผลข้างเคียงที่พบบ่อยของ statins ตามรายงานล่าสุดคืออะไร',
]

results = []
for q in DEMO_QUERIES:
    print('═'*90); print(f'❓ {q}\n')
    t0 = time.time()
    resp = agent.answer(q)
    dt = time.time() - t0
    print(f'🤖 {resp.answer}\n')
    print(f'📎 Citations: {len(resp.citations)} sources · ⏱ {dt:.1f}s\n')
    results.append({'query': q, 'answer': resp.answer, 'latency': dt})


## Step 5 — ⏱ Latency breakdown



In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3.5))
labels = [f'Q{i+1}' for i in range(len(results))]
latencies = [r['latency'] for r in results]
bars = ax.bar(labels, latencies, color='#c0392b')
ax.axhline(sum(latencies)/len(latencies), ls='--', color='gray', label=f'mean={sum(latencies)/len(latencies):.1f}s')
ax.set_ylabel('seconds'); ax.set_title('End-to-end RAG latency per query'); ax.legend()
for b, v in zip(bars, latencies):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05, f'{v:.1f}s', ha='center')


## Step 6 — 🌍 Coverage heatmap

Which sources cover which categories? Useful to show the customer that


In [ ]:
import pandas as pd, seaborn as sns, sqlite3

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql('SELECT source_name, category FROM records', conn)
conn.close()

pivot = df.groupby(['source_name','category']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(pivot, annot=True, fmt='d', cmap='Reds', cbar_kws={'label':'records'}, ax=ax)
ax.set_title('Source × Category coverage')


## 🎉 Done

You've just shown the full RAG pipeline. Next steps for the deal:

- Show the live [HF Space demo](https://huggingface.co/spaces/YOUR_USERNAME/cv-intel-rag)
- Share this notebook (outputs visible) as a PDF leave-behind
